# TEMA + MACD BTC — Research Pipeline

**Goal:** high Sharpe, **lower drawdown** via:

1. Wide **grid search** (random subsample of parameter space)
2. **Boruta** feature selection on train
3. **Optuna** TPE refinement on validation
4. **Parameter sensitivity** heatmaps around the Optuna winner
5. **Holdout test** (never used during tuning)

Composite objective (validation):

`objective = 1.0 × Sharpe + 0.35 × Calmar − 2.5 × |max_drawdown|`

> ⚠️ In-sample tuning can overfit. Treat holdout metrics as the honest score.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from strategies.tema_macd_ensemble_optimize import (
    load_btc_close,
    split_series,
    run_boruta_selection,
    wide_grid_search,
    run_optuna_study,
    config_from_optuna_params,
    parameter_sensitivity,
    evaluate_holdout,
    run_full_pipeline,
)

plt.style.use('dark_background')
sns.set_theme(style='darkgrid')


In [ ]:
# --- Config ---
START = '2018-01-01'
BORUTA_ITER = 100
GRID_MAX = 2500      # wide grid (random sample)
OPTUNA_TRIALS = 150

close = load_btc_close(START)
train, val, test = split_series(close, train_ratio=0.6, val_ratio=0.2)
print(f'Total {len(close)} | train {len(train)} | val {len(val)} | test {len(test)}')


In [ ]:
# 1) Boruta feature selection (train only)
selected, boruta_rank = run_boruta_selection(train, max_iter=BORUTA_ITER)
print('Selected features:', selected)
boruta_rank.head(20)


In [ ]:
# 2) Wide grid search on validation
grid_df = wide_grid_search(train, val, close, selected, max_combos=GRID_MAX)
print('Top 10 grid configs:')
grid_df.head(10)[['tema_period','macd_fast','macd_slow','macd_signal','long_threshold','flat_threshold','sharpe','max_drawdown','calmar','objective']]


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(grid_df['max_drawdown'], grid_df['sharpe'], c=grid_df['objective'], cmap='viridis', alpha=0.5, s=12)
ax.set_xlabel('Max drawdown (validation)')
ax.set_ylabel('Sharpe (validation)')
ax.set_title('Wide grid — Sharpe vs drawdown (color = objective)')
plt.colorbar(ax.collections[0], ax=ax, label='objective')
plt.tight_layout()
plt.show()


In [ ]:
# 3) Optuna refinement
study, trials_df = run_optuna_study(train, val, close, selected, n_trials=OPTUNA_TRIALS)
best_cfg = config_from_optuna_params(study.best_params)
print('Best params:', study.best_params)
print('Best validation objective:', study.best_value)
best_cfg


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
if not trials_df.empty:
    axes[0].plot(trials_df['number'], trials_df['value'], marker='o', ms=3, alpha=0.7)
    axes[0].set_title('Optuna trial objective')
    axes[0].set_xlabel('trial')
    if 'user_attrs_sharpe' in trials_df.columns:
        axes[1].scatter(trials_df['user_attrs_max_drawdown'], trials_df['user_attrs_sharpe'], c=trials_df['value'], cmap='plasma', s=18)
        axes[1].set_xlabel('max drawdown')
        axes[1].set_ylabel('sharpe')
        axes[1].set_title('Optuna trials')
plt.tight_layout()
plt.show()


In [ ]:
# 4) Parameter sensitivity around Optuna winner (train+val)
sens = parameter_sensitivity(pd.concat([train, val]), best_cfg, selected, tema_delta=16, th_delta=0.15)
pivot = sens.pivot_table(index='tema_period', columns='long_threshold', values='objective')
plt.figure(figsize=(10,6))
sns.heatmap(pivot, annot=False, cmap='mako')
plt.title('Sensitivity: objective by TEMA period × long threshold')
plt.tight_layout()
plt.show()


In [ ]:
# 5) Holdout evaluation (test set — untouched)
holdout = evaluate_holdout(close, test, best_cfg, selected)
val_best = grid_df.iloc[0].to_dict() if not grid_df.empty else {}

pd.DataFrame([
    {'split': 'validation_best_grid', 'sharpe': val_best.get('sharpe'), 'max_drawdown': val_best.get('max_drawdown'), 'objective': val_best.get('objective')},
    {'split': 'validation_optuna', 'sharpe': study.best_trial.user_attrs.get('sharpe'), 'max_drawdown': study.best_trial.user_attrs.get('max_drawdown'), 'objective': study.best_value},
    {'split': 'holdout_test', **holdout},
]).set_index('split')


In [ ]:
# Optional: one-shot full pipeline helper
# results = run_full_pipeline(start=START, boruta_iter=BORUTA_ITER, grid_max=GRID_MAX, optuna_trials=OPTUNA_TRIALS)


## 6) Multi-objective Optuna (Pareto: Sharpe ↑, |DD| ↓)

In [ ]:
from strategies.tema_macd_ensemble_optimize import run_optuna_multiobjective, config_from_optuna_params, pick_pareto_knee

mo_study, mo_trials, picked = run_optuna_multiobjective(train, val, close, selected, n_trials=OPTUNA_TRIALS)
pareto_cfg = config_from_optuna_params(picked['params'])
print('Pareto knee pick:', picked)
pareto_cfg


In [ ]:
# Pareto front visualization
import matplotlib.pyplot as plt
front = mo_study.best_trials
xs = [t.values[1] for t in front if t.values]
ys = [t.values[0] for t in front if t.values]
plt.figure(figsize=(8,5))
plt.scatter(xs, ys, c='cyan', s=30, label='Pareto front')
plt.xlabel('|Max DD| (minimize)')
plt.ylabel('Sharpe (maximize)')
plt.title('Multi-objective Optuna — validation Pareto front')
plt.legend()
plt.tight_layout()
plt.show()


## 7) Walk-forward OOS folds

In [ ]:
from strategies.tema_macd_ensemble_optimize import walk_forward_validate

wf = walk_forward_validate(close, pareto_cfg, selected, n_splits=5)
wf


## 8) Export live config for Vercel terminal

In [ ]:
from strategies.tema_macd_ensemble_optimize import export_live_config_from_research

# Or run: python3 scripts/export_tema_macd_live_config.py
out = export_live_config_from_research(
    start=START,
    boruta_iter=BORUTA_ITER,
    optuna_trials=OPTUNA_TRIALS,
    walk_forward_splits=5,
)
print('Live config written to:', out)
